In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/sidewalk_hazard_dataset.zip -d /content/dataset


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install cython
!pip install pycocotools
!pip install tqdm

In [ ]:
!pip install --upgrade sympy
!pip install --upgrade torch torchvision torchaudio

In [ ]:
import os
import json
import torch
from tqdm import tqdm
import torch.utils.data
from PIL import Image
from pycocotools.coco import COCO
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as T

class CustomDataset(torch.utils.data.Dataset): # 1. Custom Dataset Class
    def __init__(self, root, annotation_file, transforms=None):
        self.root = root
        self.coco = COCO(annotation_file)
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __getitem__(self, index):
        coco = self.coco
        img_id = self.ids[index]
        ann_ids = coco.getAnnIds(imgIds=img_id)
        coco_annotation = coco.loadAnns(ann_ids)
        path = coco.loadImgs(img_id)[0]['file_name']

        img = Image.open(os.path.join(self.root, path)).convert("RGB")    # Open image

        num_objs = len(coco_annotation)  # Format annotations for PyTorch
        boxes = []
        labels = []
        for i in range(num_objs):
            xmin = coco_annotation[i]['bbox'][0]
            ymin = coco_annotation[i]['bbox'][1]
            xmax = xmin + coco_annotation[i]['bbox'][2]
            ymax = ymin + coco_annotation[i]['bbox'][3]
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(coco_annotation[i]['category_id'])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        img_id = torch.tensor([img_id])

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = img_id

        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.ids)


def get_model(num_classes):  # 2. Model Setup
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")      # Load a pre-trained model
    in_features = model.roi_heads.box_predictor.cls_score.in_features  # Get the number of input features for the classifier
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes) # Replace the pre-trained head with a new one
    return model

def train(epochs=5, DATA_ROOT="/content/dataset"):  # 3. Training Logic

    train_img_dir = os.path.join(DATA_ROOT, "images/train") # 1. SETUP PATHS
    train_ann = os.path.join(DATA_ROOT, "annotations/train.json")

    with open(os.path.join(DATA_ROOT, "class_map.json"), 'r') as f:
        data = json.load(f)
        class_to_id = data["class_to_id"]       # Load class map
    num_classes = len(class_to_id) + 1

    transforms = T.Compose([T.ToTensor()])   # 2. DATA LOADERS
    dataset = CustomDataset(train_img_dir, train_ann, transforms)

    data_loader = torch.utils.data.DataLoader(  # We use batch_size=4 to speed things up if the GPU allows
        dataset, batch_size=4,
        shuffle=True,
        collate_fn=lambda x: tuple(zip(*x)),
        num_workers=2 # Uses multiple CPU cores to load images faster
    )

    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    model = get_model(num_classes)   # 3. MODEL & OPTIMIZER
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    print(f"Starting training on {len(dataset)} images...")    # 4. TRAINING LOOP
    model.train()

    model.load_state_dict(torch.load("/content/drive/MyDrive/faster_rcnn_epoch_5.pth", map_location=device))
    for epoch in range(5, epochs):
        pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{epochs}")   # Initialize tqdm progress bar
        epoch_loss = 0

        for i, (images, targets) in enumerate(pbar):
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()

            if i % 10 == 0: # Update the progress bar with the current loss every step
                pbar.set_postfix(loss=f"{losses.item():.4f}")

        avg_loss = epoch_loss / len(data_loader)
        print(f"Finished Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

        checkpoint_path = f"/content/drive/MyDrive/faster_rcnn_epoch_{epoch+1}.pth" # 5. SAVE CHECKPOINT TO DRIVE
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Saved checkpoint to: {checkpoint_path}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image
import torchvision.transforms as T
import json

def visualize_prediction(image_path, model_path, class_map_json):
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Running inference on: {device}")

    with open(class_map_json, 'r') as f:   # Load class names
        data = json.load(f)
        id_to_class = {v: k for k, v in data["class_to_id"].items()}

    num_classes = len(id_to_class) + 1
    model = get_model(num_classes)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    img = Image.open(image_path).convert("RGB")
    img_tensor = T.ToTensor()(img).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model(img_tensor)

    fig, ax = plt.subplots(1, figsize=(12, 9))
    ax.imshow(img)

    boxes = prediction[0]['boxes'].cpu().numpy()  # Move data back to CPU for plotting
    labels = prediction[0]['labels'].cpu().numpy()
    scores = prediction[0]['scores'].cpu().numpy()

    for box, label, score in zip(boxes, labels, scores):
        if score > 0.5:
            xmin, ymin, xmax, ymax = box
            rect = patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                                     linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)

            label_name = id_to_class.get(label, "Unknown")
            plt.text(xmin, ymin, f"{label_name}: {score:.2f}",
                     color='white', verticalalignment='top',
                     bbox={'color': 'red', 'alpha': 0.5, 'pad': 0})

    plt.axis('off')
    plt.show()

In [ ]:
import os
import random

def test_on_val_set(num_samples=5):
    val_img_dir = "/content/dataset/images/val"
    model_path = "/content/drive/MyDrive/faster_rcnn_epoch_6.pth"
    class_json = "/content/dataset/class_map.json"

    all_images = [f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]   # Get all images in the validation folder

    samples = random.sample(all_images, min(num_samples, len(all_images))) # Pick random samples

    print(f"Testing on {len(samples)} random validation images...")

    for img_name in samples:
        img_path = os.path.join(val_img_dir, img_name)
        print(f"Processing: {img_name}")
        visualize_prediction(img_path, model_path, class_json)

test_on_val_set(5)

In [ ]:
import time
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
import torchvision.transforms as T

def benchmark_inference_speed(model_path, img_dir, num_samples=100):
    # 1. Setup Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Benchmarking on: {device}")

    # 2. Load Model
    # Ensure num_classes matches your class_map (usually 17 for 16 classes + background)
    with open("/content/dataset/class_map.json", 'r') as f:
        class_data = json.load(f)
        num_classes = len(class_data["class_to_id"]) + 1

    model = get_model(num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()

    # 3. Prepare Images
    img_names = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))][:num_samples]
    transform = T.Compose([T.ToTensor()])

    latencies = []

    # 4. Warm-up (Important!)
    print("Warming up the GPU...")
    dummy_img = torch.randn(1, 3, 800, 800).to(device)  # GPUs are often slow on the very first image; we discard the first 5 runs.
    for _ in range(5):
        _ = model(dummy_img)

    # 5. Benchmark Loop
    print(f"Measuring latency over {len(img_names)} images...")
    with torch.no_grad():
        for name in tqdm(img_names):
            img_path = os.path.join(img_dir, name)
            img = Image.open(img_path).convert("RGB")
            img_tensor = transform(img).unsqueeze(0).to(device)

            # Start Timer
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = time.perf_counter()

            # Run Inference
            _ = model(img_tensor)

            # End Timer
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end_time = time.perf_counter()

            latencies.append(end_time - start_time)

    # 6. Report Results
    avg_latency = np.mean(latencies) * 1000 # Convert to ms
    std_latency = np.std(latencies) * 1000
    fps = 1 / np.mean(latencies)

    print("\n" + "="*30)
    print(f"INFERENCE PERFORMANCE REPORT")
    print("="*30)
    print(f"Average Latency: {avg_latency:.2f} ms")
    print(f"Std Deviation:   {std_latency:.2f} ms")
    print(f"Frames Per Second: {fps:.2f} FPS")
    print("="*30)

# Run benchmark on Epoch 5
benchmark_inference_speed(
     model_path="/content/drive/MyDrive/faster_rcnn_epoch_5.pth",
     img_dir="/content/dataset/images/val",
     num_samples=100
)

Benchmarking on: cuda
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 110MB/s]


Warming up the GPU...
Measuring latency over 100 images...


100%|██████████| 100/100 [00:13<00:00,  7.47it/s]


INFERENCE PERFORMANCE REPORT
Average Latency: 103.73 ms
Std Deviation:   7.68 ms
Frames Per Second: 9.64 FPS
